In [1]:
# ============================================================
# Gold: capa de negocio para generación EERSA
# - Lee de silver_generacion_diaria
# - Genera 4 tablas listas para BI
# ============================================================

from pyspark.sql import functions as F

# Leer Silver
df_silver = spark.sql("SELECT * FROM lh_silver_eersa.dbo.silver_generacion_diaria")
print(f"Silver input: {df_silver.count()} filas")

# ============================================================
# 1. gold_dim_planta — dimensión de plantas
# ============================================================
plantas_data = [
    ("ALAO", "generacion_propia", "Hidroeléctrica Alao", "hidroelectrica", 10.0),
    ("RIO BLANCO", "generacion_propia", "Hidroeléctrica Río Blanco", "hidroelectrica", 3.0),
    ("C.NIZAG", "generacion_propia", "Central Nizag", "hidroelectrica", 1.0),
    ("S.N.I.", "interconexion", "Sistema Nacional Interconectado", "interconexion", None),
    ("TOTAL EERSA", "agregado", "Agregado total EERSA", "agregado", None),
    ("ENERGIA ENTREGADA AL MEM (kWh)", "flujo_mem", "Energía entregada al MEM", "flujo_mem", None),
    ("ENERGIA RECIBIDA DEL MEM (kWh)", "flujo_mem", "Energía recibida del MEM", "flujo_mem", None),
]
df_dim_planta = spark.createDataFrame(
    plantas_data,
    ["planta_codigo", "tipo_fuente", "planta_nombre", "tecnologia", "capacidad_mw_aprox"]
)
df_dim_planta.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("gold_dim_planta")
print("✓ gold_dim_planta")

# ============================================================
# 2. gold_fct_generacion_diaria — pivot wide para BI
# ============================================================
df_fct = df_silver.groupBy("fecha", "planta", "tipo_fuente", "anio", "mes") \
                  .pivot("metrica").agg(F.first("valor"))

mapeo_cols = {
    "KW": "kw",
    "E.Bruta kWh": "e_bruta_kwh",
    "C.Int kWh": "c_interno_kwh",
    "E.Neta kWh": "e_neta_kwh",
    "KWh": "kwh",
    "KW-H": "kwh_alt"
}
for old, new in mapeo_cols.items():
    if old in df_fct.columns:
        df_fct = df_fct.withColumnRenamed(old, new)

df_fct.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("gold_fct_generacion_diaria")
print(f"✓ gold_fct_generacion_diaria: {df_fct.count()} filas")

# ============================================================
# 3. gold_agg_generacion_mensual — KPIs por planta y mes
# ============================================================
df_e_neta = df_silver.filter(F.col("metrica") == "E.Neta kWh")
df_kw = df_silver.filter(F.col("metrica") == "KW")

df_agg_mensual = df_e_neta.groupBy("anio", "mes", "planta", "tipo_fuente").agg(
    F.sum("valor").alias("e_neta_total_kwh"),
    F.avg("valor").alias("e_neta_promedio_diario_kwh"),
    F.count("valor").alias("dias_con_datos")
)

df_kw_max = df_kw.groupBy("anio", "mes", "planta").agg(
    F.max("valor").alias("demanda_maxima_kw"),
    F.avg("valor").alias("demanda_promedio_kw")
)

df_agg_mensual = df_agg_mensual.join(df_kw_max, ["anio", "mes", "planta"], "left")
df_agg_mensual = df_agg_mensual.withColumn(
    "factor_carga_pct",
    F.round((F.col("demanda_promedio_kw") / F.col("demanda_maxima_kw")) * 100, 2)
)

df_agg_mensual.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("gold_agg_generacion_mensual")
print(f"✓ gold_agg_generacion_mensual: {df_agg_mensual.count()} filas")

# ============================================================
# 4. gold_agg_balance_energetico_mensual — propia vs interconexión
# ============================================================
df_balance = df_e_neta.groupBy("anio", "mes", "tipo_fuente").agg(
    F.sum("valor").alias("e_neta_total_kwh")
).groupBy("anio", "mes").pivot("tipo_fuente").agg(F.first("e_neta_total_kwh"))

for col in df_balance.columns:
    if col not in ("anio", "mes"):
        df_balance = df_balance.withColumnRenamed(col, f"e_neta_{col}_kwh")

if "e_neta_generacion_propia_kwh" in df_balance.columns and "e_neta_interconexion_kwh" in df_balance.columns:
    df_balance = df_balance.withColumn(
        "demanda_total_kwh",
        F.coalesce(F.col("e_neta_generacion_propia_kwh"), F.lit(0)) +
        F.coalesce(F.col("e_neta_interconexion_kwh"), F.lit(0))
    ).withColumn(
        "cobertura_propia_pct",
        F.round((F.col("e_neta_generacion_propia_kwh") / F.col("demanda_total_kwh")) * 100, 2)
    )

df_balance.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("gold_agg_balance_energetico_mensual")
print(f"✓ gold_agg_balance_energetico_mensual: {df_balance.count()} filas")

print("\n=== GOLD COMPLETO ===")
display(df_balance.orderBy("anio", "mes"))


StatementMeta(, 3f9685b3-5870-4a4d-b787-cc0b14c3ca13, 3, Finished, Available, Finished, False)

Silver input: 5840 filas
✓ gold_dim_planta
✓ gold_fct_generacion_diaria: 1825 filas
✓ gold_agg_generacion_mensual: 36 filas
✓ gold_agg_balance_energetico_mensual: 12 filas

=== GOLD COMPLETO ===


SynapseWidget(Synapse.DataFrame, 6e5da278-0c63-4270-82d0-af8843e154f2)